# SOXX 2019–2022 原始披露集中度 Qlib 衔接

固定 `plan.json`：复用两份旧表单值和 13 份 NPORT 原件值，在 Stage1 原 Qlib 日历上按申报日期之后的下一个交易日映射；完整保留未知前缀。来源映射不读收益、不拟合、不建账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2019_2022_concentration_bridge_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
s=plan['sources'];assert sha(root/s['legacy_numeric_result'])==s['legacy_numeric_result_sha256'] and sha(root/s['NPORT_summary'])==s['NPORT_summary_sha256']
legacy=json.loads((root/s['legacy_numeric_result']).read_text());prior=json.loads((root/s['NPORT_summary']).read_text())
assert legacy['status']=='two_2019_transition_original_SOXX_reported_value_sources_closed' and len(legacy['reports'])==2
assert prior['recorder_id']==s['NPORT_recorder_id'] and prior['nas_evidence']['sha256']==s['NPORT_recorder_NAS_sha256'] and prior['nas_readback']['verified']
state={'status':'started','new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2019-01-02',end_time='2023-01-10',freq='day'))
with R.uri_context(prior['recorder_uri']):
 source_rec=R.get_recorder(recorder_id=prior['recorder_id'],experiment_id=prior['experiment_id'])
 nport=source_rec.load_object('analysis/source_events')
assert len(nport)==13
legacy_events=pd.DataFrame([{'accession':x['accession'],'report_period':x['report_period'],'filed':x['filed'],'form':'legacy_old_form','top5':x['top5_disclosed_equity_value_share'],'HHI':x['herfindahl_sum_squared_disclosed_equity_weights'],'source_sha256':x['raw_sha256']} for x in legacy['reports']])
nport_events=pd.DataFrame([{'accession':x['accession'],'report_period':x['report_period'],'filed':x['filed'],'form':'NPORT-P','top5':x['top5_disclosed_EC_value_share'],'HHI':x['HHI_disclosed_EC_value'],'source_sha256':x['source_xml_sha256']} for _,x in nport.iterrows()])
events=pd.concat([legacy_events,nport_events],ignore_index=True).sort_values('filed').reset_index(drop=True)
assert len(events)==15 and events.accession.is_unique and events.report_period.is_unique
events['filed_date']=pd.to_datetime(events.filed)
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
signals=calendar[(calendar>=pd.Timestamp(plan['calendar_span'][0]))&(calendar<=pd.Timestamp(plan['calendar_span'][1]))]
mapped=pd.merge_asof(pd.DataFrame({'signal_date':signals}),events[['effective_signal_date','accession','report_period','filed_date','form','top5','HHI']],left_on='signal_date',right_on='effective_signal_date',direction='backward')
mapped['known']=mapped.accession.notna();mapped['source_age_days']=(mapped.signal_date-mapped.filed_date).dt.days
assert mapped.signal_date.is_unique and len(mapped)==len(signals) and mapped.loc[mapped.known,'source_age_days'].ge(1).all()
assert not mapped.loc[mapped.signal_date<events.effective_signal_date.min(),'known'].any()
frame=mapped.set_index('signal_date')[['top5','HHI','known','source_age_days']]
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5','HHI']],'meta':frame[['known','source_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'late_train':tuple(plan['calendar_span'])})
prepared=dataset.prepare('late_train',col_set=['feature','meta']);assert len(prepared)==len(signals)
assert int(mapped.known.sum())==int(prepared[('meta','known')].sum())
audit={'status':'legacy_plus_NPORT_15_original_report_events_Qlib_mapped_source_only','source_plan_sha256':sha(card/'plan.json'),'report_events':len(events),'legacy_events':len(legacy_events),'NPORT_events':len(nport_events),'late_train_Qlib_sessions':len(signals),'known_signal_sessions':int(mapped.known.sum()),'unknown_prefix_sessions':int((~mapped.known).sum()),'first_effective_signal':str(events.effective_signal_date.min().date()),'last_effective_signal':str(events.effective_signal_date.max().date()),'max_source_age_days':int(mapped.source_age_days.max()),'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'full_Stage1_training_coverage_qualified':False}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('late_train',col_set=['feature','meta']))==len(signals)
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['question'][:80],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'legacy':s['legacy_numeric_result_sha256'],'NPORT':s['NPORT_summary_sha256']})
 rec.save_objects(artifact_path='analysis',source_events=events,mapped_signals=mapped,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=len(events),late_train_sessions=len(signals),known_source_sessions=int(mapped.known.sum()),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 assert len(checked.load_object('metadata/dataset').prepare('late_train',col_set=['feature','meta']))==len(signals)
 assert len(checked.load_object('analysis/source_events'))==15
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_2019_2022_concentration_bridge','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-transition-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'2019_2022_late_train_original_reported_concentration_source_bridge_qualified_early_history_pending','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','report_events':15,'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:audit[k] for k in ('report_events','late_train_Qlib_sessions','known_signal_sessions','unknown_prefix_sessions','first_effective_signal','last_effective_signal','max_source_age_days')});print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'report_events': 15, 'late_train_Qlib_sessions': 1008, 'known_signal_sessions': 967, 'unknown_prefix_sessions': 41, 'first_effective_signal': '2019-03-04', 'last_effective_signal': '2022-11-29', 'max_source_age_days': 171}
{'recorder_id': 'c6b3030ff32445e2b5b24b9e9e404524', 'nas_sha256': '199179d4f1c21543a306c9afa21ee4027a306a101a4c62973f57892fdb273c86'}


## 中文结论

两份旧表单和 13 份 NPORT 原件共 15 个来源事件映射到原 Stage1 Qlib 日历。2019–2022 年 1,008 日中 967 日已有此前申报的集中度代理，首个 2019-03-04 生效前 41 日保持未知；最大申报日龄 171 天。DatasetH/Recorder `c6b3030ff32445e2b5b24b9e9e404524` 与 NAS SHA-256 `199179d4f1c21543a306c9afa21ee4027a306a101a4c62973f57892fdb273c86` 回读通过。0 收益读取、0 拟合、0 账户。此为滞后披露代理而非指数逐日权重，完整 Stage1 来源和 0bp SOXX 门尚未完成。